# Confirmed-config held-out FlowEdit regeneration (controlled, resume-safe)

This is a **new**, private Drive run, not a modification of frozen N07/N17. All 12 known tuning sources are excluded from generated comparisons. Three conditions on the same 108 sources, seed 42: N07 target g4.5, N07 target g5.5, and a **same facial expression** no-emotion target g4.5 (324 images total). Historical source/target prompts, frozen FlowEdit and SD3 revision are verified. Outputs/receipts are written immediately to private Drive. **Generation is OFF by default.** 324 generations may be expensive.

**Important:** This runner verifies generation only. No new accuracy, biometric coverage or NIQE claim is valid before the corrected evaluator is applied to these outputs. In particular, do not use the old YOLO→FaceAnalysis evaluator with low coverage as a final privacy estimate.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
from pathlib import Path
import os, sys, json, hashlib, subprocess, time, shutil, re
import pandas as pd, yaml
from PIL import Image

RUN_GENERATION = True  # run-enabled; completed 324 verified receipts will resume without regeneration
ROOT=Path('/content/drive/MyDrive/thesis_all/outputs/runs')
N07=ROOT/'20260818T210720Z_notebook07_affectnet_larger_6aeb3643_f1142664'
OUT=ROOT/'nb23_final_heldout_regeneration_frozen_v1'
FLOW_COMMIT='85594626161caca9dafa68592c1ffa21db87a806'
SD3_REV='ea42f8cef0f178587cf766dc8129abd379c90671'
MODEL='stabilityai/stable-diffusion-3-medium-diffusers'
TUNING={'source_000','source_001','source_005','source_006','source_010','source_011','source_015','source_016','source_020','source_021','source_025','source_026'}
SAME_EXPR_TARGET=('A realistic close-up portrait photograph of a different person '
 'with the same facial expression as the source person, keeping similar head '
 'angle and eye gaze to the original. Photorealistic.')
SEED=42
for folder in ('images','configs','logs','staging','metadata'):
 (OUT/folder).mkdir(parents=True,exist_ok=True)
def sha(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for chunk in iter(lambda:f.read(1048576),b''):h.update(chunk)
 return h.hexdigest()
def valid_png(path):
 try:
  with Image.open(path) as im: im.verify()
  with Image.open(path) as im: return im.size==(512,512)
 except Exception: return False
print('Run folder:',OUT, '| generation enabled:',RUN_GENERATION)

In [ ]:
sources=pd.read_csv(N07/'selected_sources.csv')
assert len(sources)==120 and sources.source_id.is_unique and sources.emotion.nunique()==6
assert TUNING.issubset(set(sources.source_id)) and len(TUNING)==12
held=sources.loc[~sources.source_id.isin(TUNING)].sort_values('source_id')
assert len(held)==108 and held.source_id.is_unique
jobs=[]
for x in held.itertuples(index=False):
 sid=x.source_id
 src=N07/'sources'/f'{sid}.png'
 assert src.is_file() and sha(src)==x.prepared_source_sha256
 cfg=N07/'flowedit_configs'/f'nb07_{sid}_flowedit_g5p5_s42_f1142664_edits.yaml'
 assert cfg.exists(),f'Missing frozen N07 YAML: {cfg}'
 frozen=yaml.safe_load(cfg.read_text())
 assert isinstance(frozen,list) and len(frozen)==1
 source_prompt=frozen[0]['source_prompt'];target=frozen[0]['target_prompts'][0]
 assert str(x.emotion).lower() in source_prompt.lower() and str(x.emotion).lower() in target.lower()
 assert 'different person' in target.lower()
 assert not any(re.search(r'\\b'+e+r'\\b',SAME_EXPR_TARGET.lower())
                for e in ('happy','sad','fear','angry','disgust','surprise','neutral'))
 for condition,g,target_text in [('n07_g4p5',4.5,target),('n07_g5p5',5.5,target),
                                  ('same_expression_g4p5',4.5,SAME_EXPR_TARGET)]:
  name=f'nb23_{sid}_{condition}_s{SEED}'
  jobs.append(dict(name=name,source_id=sid,emotion=x.emotion,
   source_path=str(src),source_sha256=x.prepared_source_sha256,
   source_prompt=source_prompt,target_prompt=target_text,
   condition=condition,tar_guidance=g,seed=SEED))
assert len(jobs)==324 and len({x['name'] for x in jobs})==324
protocol={'flowedit_commit':FLOW_COMMIT,'sd3_revision':SD3_REV,'T_steps':40,
 'src_guidance_scale':3.5,'n_min':0,'n_max':33,'n_avg':1,'seed':42,
 'excluded_tuning':sorted(TUNING),'jobs':jobs}
p=OUT/'metadata'/'private_frozen_protocol.json'
if p.exists(): assert json.loads(p.read_text())==protocol,'STOP: frozen protocol changed'
else: p.write_text(json.dumps(protocol,indent=2))
print('PREFLIGHT PASS: 108 held-out sources, 324 planned images')
print(pd.Series([j['condition'] for j in jobs]).value_counts())

In [ ]:
if RUN_GENERATION:
 import torch
 assert torch.cuda.is_available(),'Select GPU in Colab before generation'
 subprocess.run([sys.executable,'-m','pip','-q','install','diffusers==0.30.1',
  'transformers==4.44.2','accelerate','sentencepiece','protobuf',
  'safetensors','huggingface_hub==0.36.2'],check=True)
 from huggingface_hub import HfApi,login
 token=userdata.get('HF_TOKEN')
 assert token,'Set HF_TOKEN in Colab Secrets and enable notebook access'
 login(token=token,add_to_git_credential=False)
 assert HfApi().model_info(repo_id=MODEL,revision=SD3_REV,token=token).sha==SD3_REV
 WORK=Path('/content/flowedit_nb23')
 if not WORK.exists():
  subprocess.run(['git','clone','-q','https://github.com/fallenshock/FlowEdit.git',str(WORK)],check=True)
 subprocess.run(['git','-C',str(WORK),'checkout','-q',FLOW_COMMIT],check=True)
 assert subprocess.check_output(['git','-C',str(WORK),'rev-parse','HEAD'],text=True).strip()==FLOW_COMMIT
 original=subprocess.check_output(['git','-C',str(WORK),'show',
  f'{FLOW_COMMIT}:run_script.py'],text=True)
 needle='StableDiffusion3Pipeline.from_pretrained("stabilityai/stable-diffusion-3-medium-diffusers", torch_dtype=torch.float16)'
 replacement='StableDiffusion3Pipeline.from_pretrained("stabilityai/stable-diffusion-3-medium-diffusers", revision="'+SD3_REV+'", torch_dtype=torch.float16)'
 assert original.count(needle)==1,'STOP: frozen upstream runner unexpectedly changed'
 patched=original.replace(needle,replacement)
 (WORK/'run_script.py').write_text(patched)
 assert patched.count('revision="'+SD3_REV+'"')==1
 (WORK/'example_images').mkdir(exist_ok=True)
 (OUT/'metadata'/'environment.json').write_text(json.dumps(
  dict(flowedit_commit=FLOW_COMMIT,sd3_revision=SD3_REV,
   runner_original_sha256=hashlib.sha256(original.encode()).hexdigest(),
   runner_patched_sha256=hashlib.sha256(patched.encode()).hexdigest(),
   torch=torch.__version__,gpu=torch.cuda.get_device_name(0)),indent=2))
 print('MODEL COMMIT / REVISION VERIFIED')
else:
 print('PREVIEW MODE: no model download or generation')

In [ ]:
def output_for_run(name):
 root=WORK/'outputs'/name/'SD3'/('src_'+name)/'tar_0'
 files=list(root.glob('output_*.png')) if root.is_dir() else []
 assert len(files)==1,f'Expected one generated PNG for {name}, found {len(files)}'
 return files[0]

if RUN_GENERATION:
 for i,j in enumerate(jobs,1):
  name=j['name']
  final=OUT/'images'/f'{name}.png'
  receipt=OUT/'images'/f'{name}.receipt.json'
  if final.exists() or receipt.exists():
   assert final.is_file() and receipt.is_file(),'STOP: incomplete persisted job'
   saved=json.loads(receipt.read_text())
   assert saved['output_sha256']==sha(final) and valid_png(final)
   assert saved['source_sha256']==j['source_sha256']
   assert saved['runner_commit']==FLOW_COMMIT and saved['sd3_revision']==SD3_REV
   assert saved['condition']==j['condition']
   print(f'[{i}/324] VERIFIED RESUME {name}',flush=True)
   continue
  assert sha(j['source_path'])==j['source_sha256']
  shutil.copy2(j['source_path'],WORK/'example_images'/f'{name}.png')
  edit_path=OUT/'configs'/f'{name}_edits.yaml'
  exp_path=OUT/'configs'/f'{name}_exp.yaml'
  edits=[{'input_img':f'example_images/{name}.png',
   'source_prompt':j['source_prompt'],
   'target_prompts':[j['target_prompt']],'target_codes':['different_person']}]
  exp=[{'exp_name':name,'dataset_yaml':str(edit_path),
   'model_type':'SD3','sampler_type':'FlowEditSD3','T_steps':40,
   'src_guidance_scale':3.5,'tar_guidance_scale':j['tar_guidance'],
   'n_min':0,'n_max':33,'n_avg':1,'seed':SEED}]
  edit_path.write_text(yaml.safe_dump(edits,sort_keys=False))
  exp_path.write_text(yaml.safe_dump(exp,sort_keys=False))
  logfile=OUT/'logs'/f'{name}.txt'
  start=time.time()
  print(f'[{i}/324] GENERATING {name}',flush=True)
  with logfile.open('w') as stream:
   run=subprocess.run([sys.executable,'run_script.py','--exp_yaml',str(exp_path)],
    cwd=WORK,env={**os.environ,'HF_TOKEN':token},stdout=stream,stderr=subprocess.STDOUT)
  assert run.returncode==0,f'Generation failed; inspect {logfile}'
  produced=output_for_run(name)
  assert valid_png(produced),f'Invalid generated image {name}'
  staging=OUT/'staging'/f'{name}.png.tmp'
  shutil.copyfile(produced,staging)
  assert valid_png(staging)
  os.replace(staging,final)
  saved=dict(source_id=j['source_id'],emotion=j['emotion'],
   seed=SEED,condition=j['condition'],run_name=name,status='VERIFIED',
   source_sha256=j['source_sha256'],output_path=str(final),
   output_sha256=sha(final),runner_commit=FLOW_COMMIT,
   sd3_revision=SD3_REV,duration_seconds=round(time.time()-start,2))
  receipt.write_text(json.dumps(saved,indent=2))
  print(f'[{i}/324] SAVED',flush=True)
else:
 print('GENERATION DISABLED: zero new images')

In [ ]:
records=[]
for j in jobs:
 name=j['name'];p=OUT/'images'/f'{name}.png';r=OUT/'images'/f'{name}.receipt.json'
 if p.exists() or r.exists():
  assert p.is_file() and r.is_file(),f'Partial output {name}'
  record=json.loads(r.read_text())
  assert valid_png(p) and sha(p)==record['output_sha256']
  assert record['source_sha256']==j['source_sha256'] and record['condition']==j['condition']
  assert record['runner_commit']==FLOW_COMMIT and record['sd3_revision']==SD3_REV
  records.append(record)
assert len({r['run_name'] for r in records})==len(records)
if records:
 pd.DataFrame(records).to_csv(OUT/'metadata'/'private_verified_receipts.csv',index=False)
status={'status':'GENERATED_COMPLETE_EVALUATION_PENDING' if len(records)==324 else 'INCOMPLETE',
 'expected':324,'verified':len(records),'sources':108,
 'excluded_tuning':12,'corrected_biometrics':'PENDING',
 'fer_evaluation':'PENDING','niqe_evaluation':'PENDING','manual_qc':'PENDING'}
(OUT/'metadata'/'status.json').write_text(json.dumps(status,indent=2))
print(json.dumps(status,indent=2))
if RUN_GENERATION: assert len(records)==324,'Incomplete: rerun SAME notebook to resume'
print('Do NOT update PaperDock Results before corrected privacy, FER and IQA evaluation.')